# Natural Language Processing - Text Preprocessing

## Libraries and settings

In [1]:
# Libraries
import os
import re
import string
import numpy as np
import pandas as pd
from pprint import pprint

import nltk

# Import only once
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.chunk import tree2conlltags
from nltk.chunk import conlltags2tree
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Current working directory
print('Current working directory:', os.getcwd())

Current working directory: /home/runner/work/data_analytics/data_analytics/Week_11


[nltk_data] Downloading package stopwords to /home/runner/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/runner/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/runner/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/runner/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/runner/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## Defining documents

In [2]:
# Defining documents (=sentenses)
d1 = 'The car is driven on the road.'
d2 = 'The truck is driven on the highway.'
d3 = 'The bicycle is driven on the bicycle path.'

corpus_01 = d1 + ' ' + d2 + ' ' + d3
corpus_01

'The car is driven on the road. The truck is driven on the highway. The bicycle is driven on the bicycle path.'

## Text preprocessing
#### Steps:
- Text to lowercase
- Removing punctuations
- Tokenization
- Removal of stop words
- Lemmatization

### Text to lowercase

In [3]:
# Text to lowercase function
def text_lowercase(text):
    return text.lower()

# Text to lowercase
corpus_02 = text_lowercase(corpus_01)
corpus_02

'the car is driven on the road. the truck is driven on the highway. the bicycle is driven on the bicycle path.'

### Removing punctuation

In [4]:
# Remove punctuation function
def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

# Remove punctuation
corpus_03 = remove_punctuation(corpus_02)
corpus_03

'the car is driven on the road the truck is driven on the highway the bicycle is driven on the bicycle path'

### Tokenize text & removal of stopwords

In [5]:
# Show english stopwords
eng_stopwords = set(stopwords.words('english'))
print("List of english stopwords:")
print(eng_stopwords)

List of english stopwords:
{"we're", "i'll", 'as', 'too', 'haven', 'at', 'off', 'should', "mightn't", "doesn't", 'here', "isn't", 'no', 'itself', "it'd", 'wouldn', 'under', 'being', "don't", 'our', 'mustn', "you'd", 'in', 'had', "they'll", 'it', "hasn't", 'own', "hadn't", 'has', 'her', 'him', 'ma', 'again', 'about', "i've", "needn't", 'did', 'through', 'after', 'couldn', "you'll", "wasn't", 'each', 'd', 'whom', "he's", 'myself', 'those', "haven't", "shouldn't", 'up', "weren't", 'some', 'above', 's', "couldn't", "mustn't", 'than', 'of', 'i', 'there', 'few', 'same', 'his', 'm', "won't", 'my', 'only', 'when', 'where', 'shouldn', 'other', "she'll", "she's", 't', 'am', 'doing', 'from', 'was', "wouldn't", "should've", 'themselves', "i'd", 'are', 'll', 'do', 'a', 'any', 'not', 'so', 'the', 'by', "we've", 'which', 'now', 'on', "you're", 'over', 'ourselves', 're', "it'll", 'aren', 'once', "we'll", 'yours', 'have', 'such', 'that', 'won', 'down', 'if', 'during', 'most', 'himself', 'you', 'me', 'o

In [6]:
# Function for tokenization and the removal of stopwords
def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word not in stop_words]
    return filtered_text
 
# Remove stopwords
corpus_04 = remove_stopwords(corpus_03)
print(corpus_04, end="")

['car', 'driven', 'road', 'truck', 'driven', 'highway', 'bicycle', 'driven', 'bicycle', 'path']

### Lemmatization

In [7]:
# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

# Lemmatize string function
def lemmatize_word(text):
    word_tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(word, pos ='v') for word in word_tokens]
    return lemmas

# Lemmatize
lem = []
for i in corpus_04:
    lem.append(lemmatize_word(i))

# Nested list to list
corpus_05 = [' '.join([str(x) for x in lst]) for lst in lem]

print('Before lemmatization:')
print(corpus_04, '\n')

print('After lemmatization:')
print(corpus_05, end="")

Before lemmatization:
['car', 'driven', 'road', 'truck', 'driven', 'highway', 'bicycle', 'driven', 'bicycle', 'path'] 

After lemmatization:
['car', 'drive', 'road', 'truck', 'drive', 'highway', 'bicycle', 'drive', 'bicycle', 'path']

## Redefine the text corpus (pre-processed)

In [8]:
# We will use the lemmatized words above to re-define our corpus 
corpus = ['car drive road', 
          'truck drive highway', 
          'bicycle drive bicycle path']

## Document-term matrix with ngram_range=(1,1)

In [9]:
# Vectorizer with ngram_range=(1,1)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(1,1))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   bicycle  car  drive  highway  path  road  truck
0        0    1      1        0     0     1      0
1        0    0      1        1     0     0      1
2        2    0      1        0     1     0      0


## Document-term matrix with ngram_range=(2,2)

In [10]:
# Vectorizer with with ngram_range=(2,2)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(2,2))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   bicycle drive  bicycle path  car drive  drive bicycle  drive highway  \
0              0             0          1              0              0   
1              0             0          0              0              1   
2              1             1          0              1              0   

   drive road  truck drive  
0           1            0  
1           0            1  
2           0            0  


## Term frequency-inverse document frequency (TF-IDF)
- For details see: https://www.learndatasci.com/glossary/tf-idf-term-frequency-inverse-document-frequency

### Term Frequency (TF)

In [11]:
# Compute Term Frequency (TF)
words_set = set()
for doc in corpus:
    words = doc.split(' ')
    words_set = words_set.union(set(words))
    
print('Number of words in the corpus:',len(words_set), '\n')
print('The words in the corpus: \n', words_set)

# Number of documents in the corpus
n_docs = len(corpus)

# Number of unique words in the corpus 
n_words_set = len(words_set)

df_tf = pd.DataFrame(np.zeros((n_docs, n_words_set)), 
                     columns=list(words_set))

print("\nTerm Frequency (TF):")
for i in range(n_docs):
    # Words in the document
    words = corpus[i].split(' ')
    for w in words:
        df_tf[w][i] = df_tf[w][i] + (1 / len(words))
        
print(df_tf.round(4))

Number of words in the corpus: 7 

The words in the corpus: 
 {'car', 'drive', 'truck', 'highway', 'road', 'path', 'bicycle'}

Term Frequency (TF):
      car   drive   truck  highway    road  path  bicycle
0  0.3333  0.3333  0.0000   0.0000  0.3333  0.00      0.0
1  0.0000  0.3333  0.3333   0.3333  0.0000  0.00      0.0
2  0.0000  0.2500  0.0000   0.0000  0.0000  0.25      0.5


### Inverse Document Frequency (IDF)

In [12]:
# Computing Inverse Document Frequency (IDF)
print("\nInverse Document Frequency (IDF):")

idf = {}

for w in words_set:
    
    # k = number of documents that contain this word
    k = 0
    
    for i in range(n_docs):
        if w in corpus[i].split():
            k += 1
            
    idf[w] =  np.log10(n_docs / k).round(4)
    
    print(f'{w:>15}: {idf[w]:>10}')


Inverse Document Frequency (IDF):
            car:     0.4771
          drive:        0.0
          truck:     0.4771
        highway:     0.4771
           road:     0.4771
           path:     0.4771
        bicycle:     0.4771


### Term Frequency - Inverse Document Frequency (TF-IDF)

In [13]:
# Computing TF-IDF
df_tf_idf = df_tf.copy()

for w in words_set:
    for i in range(n_docs):
        df_tf_idf[w][i] = df_tf[w][i] * idf[w]

print('\nTF-IDF:')
print(df_tf_idf.round(4))


TF-IDF:
     car  drive  truck  highway   road    path  bicycle
0  0.159    0.0  0.000    0.000  0.159  0.0000   0.0000
1  0.000    0.0  0.159    0.159  0.000  0.0000   0.0000
2  0.000    0.0  0.000    0.000  0.000  0.1193   0.2386


## Part-of-Speach (POS) tagging
For meaning of POS-tags see: https://pythonexamples.org/nltk-pos-tagging

In [14]:
text = '''European authorities fined Google a record $5.1 
          billion on Wednesday for abusing its power in the 
          mobile phone market.'''

def preprocess(sent):
    sent = nltk.word_tokenize(sent)
    sent = nltk.pos_tag(sent)
    return sent

sent = preprocess(text)
pattern = 'NP: {<DT>?<JJ>*<NN>}'

cp = nltk.RegexpParser(pattern)
cs = cp.parse(sent)

iob_tagged = tree2conlltags(cs)

# Print the POS-tags
pprint(iob_tagged)

[('European', 'JJ', 'O'),
 ('authorities', 'NNS', 'O'),
 ('fined', 'VBD', 'O'),
 ('Google', 'NNP', 'O'),
 ('a', 'DT', 'B-NP'),
 ('record', 'NN', 'I-NP'),
 ('$', '$', 'O'),
 ('5.1', 'CD', 'O'),
 ('billion', 'CD', 'O'),
 ('on', 'IN', 'O'),
 ('Wednesday', 'NNP', 'O'),
 ('for', 'IN', 'O'),
 ('abusing', 'VBG', 'O'),
 ('its', 'PRP$', 'O'),
 ('power', 'NN', 'B-NP'),
 ('in', 'IN', 'O'),
 ('the', 'DT', 'B-NP'),
 ('mobile', 'JJ', 'I-NP'),
 ('phone', 'NN', 'I-NP'),
 ('market', 'NN', 'B-NP'),
 ('.', '.', 'O')]


### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [15]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.11.0-1018-azure
Datetime: 2025-12-03 10:28:05
Python Version: 3.12.3
-----------------------------------
